In [1]:
# ─── Vector DB 구성용 코드 파일 ───────────────────────────────────────
import os
import re
import time
import shutil
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# ─── 경로 설정 ───────────────────────────────────────────────
PDF_DIR            = "./data/manual_reports"  # 수동 업로드 PDF
CHROMA_PERSIST_DIR = "./chroma_db"            # 벡터 DB

# ─── 공통 상수 ───────────────────────────────────────────────
# arXiv 파일명 패턴: 2503.03182_Title...pdf
ARXIV_PATTERN = re.compile(r'^(\d{4}\.\d{4,5})_')

# 선택 수집(특허)이 비활성화 상태일 때 변수 초기화
PATENT_DOCS = []

os.makedirs(PDF_DIR, exist_ok=True)
print(f"PDF 폴더  : {os.path.abspath(PDF_DIR)}")
print(f"ChromaDB  : {os.path.abspath(CHROMA_PERSIST_DIR)}")
print("환경 설정 완료")


PDF 폴더  : /Users/gani/Desktop/ai_semi/data/manual_reports
ChromaDB  : /Users/gani/Desktop/ai_semi/chroma_db
환경 설정 완료


## [선택] arXiv 논문 자동 수집 (사용하지 않아도 됩니다)

> ⚠️ **이 셀은 선택 사항입니다.**  
> 현재 파이프라인은 `data/manual_reports/` 폴더에 직접 업로드된 PDF를 RAG 소스로 사용합니다.  
> arXiv API로 논문을 추가 수집하고 싶을 때만 아래 셀을 실행하세요.  
> 수집된 PDF는 자동으로 `data/manual_reports/`에 저장됩니다.


In [ ]:
# ⚠️ 선택 실행 셀 - 필요할 때만 실행
# data/manual_reports/ 에 직접 PDF를 넣어둔 경우 이 셀은 건너뛰세요.

RUN_ARXIV_COLLECTION = False  # True로 바꾸고 실행해야 수집 시작

if RUN_ARXIV_COLLECTION:
    import arxiv

    ARXIV_TOPICS = {
        "HBM_Stacking":          "HBM high bandwidth memory 3D stacking TSV",
        "Advanced_Packaging":    "advanced packaging chiplet heterogeneous integration",
        "PIM_CXL_Computing":     "processing-in-memory CXL near-memory computing",
        "Logic_Process_2nm_GAA": "2nm gate-all-around GAA transistor CMOS scaling",
        "EUV_Process_Equipment": "EUV extreme ultraviolet lithography semiconductor",
        "Novel_Materials":       "wide bandgap GaN SiC 2D materials semiconductor",
        "Power_Compound_Semi":   "power semiconductor GaN SiC compound device",
        "Neuromorphic_NPU_AI":   "neuromorphic computing AI accelerator NPU chip",
        "Resistive_Memory":      "RRAM ReRAM resistive memory non-volatile",
        "Silicon_Photonics":     "silicon photonics optical interconnect co-packaged",
        "Thermal_Management":    "thermal management semiconductor 3D IC heat",
        "Next_Gen_Memory_Arch":  "next generation memory architecture LPDDR GDDR",
    }

    os.makedirs(PDF_DIR, exist_ok=True)
    client = arxiv.Client()

    for topic, query in ARXIV_TOPICS.items():
        print(f"\n[{topic}] 검색 중...")
        search = arxiv.Search(query=query, max_results=5, sort_by=arxiv.SortCriterion.Relevance)
        for paper in client.results(search):
            arxiv_id = paper.entry_id.split('/')[-1]
            clean_title = re.sub(r'[^\w\s-]', '', paper.title).strip().replace(' ', '_')[:50]
            pdf_filename = f"{arxiv_id}_{clean_title}.pdf"
            pdf_path = os.path.join(PDF_DIR, pdf_filename)
            if not os.path.exists(pdf_path):
                try:
                    paper.download_pdf(filename=pdf_path)
                    print(f"  저장: {pdf_filename}")
                    time.sleep(1)
                except Exception as e:
                    print(f"  실패: {paper.title[:40]} - {e}")
            else:
                print(f"  이미 존재: {pdf_filename}")
    print("\narXiv 수집 완료")
else:
    print("[건너뜀] RUN_ARXIV_COLLECTION = False")
    print("수동 업로드 PDF를 사용합니다. (data/manual_reports/ 폴더 확인)")



📂 카테고리: [HBM_Stacking]
🔍 'HBM4 High Bandwidth Memory hybrid bonding stacking...' 검색 중...
🔍 'HBM3E DRAM thermal management power efficiency stacking...' 검색 중...
🔍 'through silicon via TSV 3D memory yield reliability...' 검색 중...
🔍 'high bandwidth memory power delivery bandwidth AI GPU...' 검색 중...

📂 카테고리: [Advanced_Packaging]
🔍 'hybrid bonding copper direct bonding wafer die pitch scaling...' 검색 중...
🔍 'chiplet heterogeneous integration CoWoS silicon interposer EMIB...' 검색 중...
🔍 'fan-out wafer level package FOWLP memory logic stacking...' 검색 중...
🔍 'die to wafer bonding 3D IC yield thermal management...' 검색 중...
🔍 'co-packaged optics CPO photonics packaging bandwidth...' 검색 중...

📂 카테고리: [PIM_CXL_Computing]
🔍 'processing in memory PIM DRAM AI accelerator bandwidth wall...' 검색 중...
🔍 'compute express link CXL memory expansion disaggregation...' 검색 중...
🔍 'near memory computing energy efficiency AI inference...' 검색 중...
🔍 'in-memory computing analog digital processing DRAM...' 검색 중...

📂

## [선택] 특허 자동 수집 (SerpAPI Google Patents)

> ⚠️ **이 셀도 선택 사항입니다.**  
> 특허 메타데이터를 추가로 수집하고 싶을 때만 실행하세요.  
> 특허는 PDF가 아닌 텍스트 청크로 직접 ChromaDB에 추가됩니다.


In [ ]:
# ⚠️ 선택 실행 셀 — 특허 수집이 필요할 때만 True로 변경 후 실행
RUN_PATENT_COLLECTION = False

if RUN_PATENT_COLLECTION:
    from serpapi import GoogleSearch

    PATENT_QUERIES = {
        "HBM_Memory":          "HBM high bandwidth memory stacking",
        "Advanced_Packaging":  "chiplet advanced packaging heterogeneous",
        "Logic_Process":       "2nm GAA gate-all-around transistor process",
        "EUV_Lithography":     "EUV extreme ultraviolet lithography",
        "Power_Semiconductor": "GaN SiC power semiconductor device",
        "Silicon_Photonics":   "silicon photonics optical interconnect",
        "AI_Accelerator":      "AI neural network accelerator chip design",
        "PIM_Computing":       "processing in memory CXL compute",
        "Novel_Materials":     "2D materials semiconductor TFET device",
        "Thermal_Management":  "3D IC thermal management heat dissipation",
    }

    SERPAPI_KEY = os.getenv("SERPAPI_API_KEY", "")
    if not SERPAPI_KEY:
        print("SERPAPI_API_KEY 없음 — .env 파일을 확인하세요.")
    else:
        PATENT_DOCS = []
        for category, query in PATENT_QUERIES.items():
            print(f"[{category}] 검색 중...")
            params  = {"engine": "google_patents", "q": query,
                       "num": 10, "api_key": SERPAPI_KEY}
            results = GoogleSearch(params).get_dict().get("organic_results", [])
            for r in results:
                title   = r.get("title", "")
                snippet = r.get("snippet", "")
                if title:                          # ← 들여쓰기 수정
                    doc = Document(
                        page_content=f"{title}\n\n{snippet}",
                        metadata={
                            "source_id":     r.get("patent_id", ""),
                            "document_type": "patent",
                            "category":      category,
                            "year":          r.get("priority_date", "")[:4],
                        }
                    )
                    PATENT_DOCS.append(doc)
        print(f"특허 {len(PATENT_DOCS)}건 수집 완료")
else:
    print("[건너뜀] RUN_PATENT_COLLECTION = False")



카테고리: [HBM_3D_Memory]
HBM4 high bandwidth memory hybrid bonding stacking Samsung Micron...' 특허 검색 중...
3D DRAM stacking copper direct bonding TSV through silicon via...' 특허 검색 중...
HBM thermal management heat spreader underfill memory...' 특허 검색 중...

카테고리: [Hybrid_Bonding_Packaging]
hybrid bonding die to wafer bonding copper interconnect...' 특허 검색 중...
chiplet heterogeneous integration interposer bridge die...' 특허 검색 중...
EMIB embedded multi-die interconnect bridge Intel...' 특허 검색 중...
Foveros 3D stacking face to face bonding Intel...' 특허 검색 중...
CoWoS chip on wafer on substrate TSMC HBM...' 특허 검색 중...

카테고리: [GAA_2nm_Logic_Process]
gate all around nanosheet GAAFET 2nm 3nm logic TSMC Intel...' 특허 검색 중...
backside power delivery BSPDN power rail logic transistor...' 특허 검색 중...
CFET complementary FET 3D stacking P-type N-type...' 특허 검색 중...
Intel 18A RibbonFET PowerVia process node...' 특허 검색 중...
Rapidus 2nm GAA Japan TSMC IBM process...' 특허 검색 중...

카테고리: [EUV_Lithography_Equipment]
EU

## [수집 현황 확인] 데이터 인벤토리

`data/manual_reports/` 폴더에 있는 모든 PDF 파일의 현황을 요약합니다.  
파일명에서 arXiv ID 패턴(`XXXX.XXXXX_`)이 감지되면 **논문**, 나머지는 **동향보고서**로 자동 분류됩니다.


In [2]:
def show_data_inventory(pdf_dir: str = PDF_DIR):
    print("=" * 65)
    print(" 데이터 인벤토리 현황 (data/manual_reports/)")
    print("=" * 65)

    if not os.path.exists(pdf_dir):
        print(f"폴더가 없습니다: {pdf_dir}")
        return

    pdf_files     = sorted([f for f in os.listdir(pdf_dir) if f.endswith(".pdf")])
    papers        = [f for f in pdf_files if ARXIV_PATTERN.match(f)]
    trend_reports = [f for f in pdf_files if not ARXIV_PATTERN.match(f)]

    print(f"\n 총 PDF     : {len(pdf_files)}건")
    print(f"   ├── arXiv 논문    : {len(papers)}건")
    print(f"   └── 동향보고서    : {len(trend_reports)}건")

    if papers:
        print(f"\n  [arXiv 논문]")
        for f in papers:
            arxiv_id   = ARXIV_PATTERN.match(f).group(1)
            title_part = f[len(ARXIV_PATTERN.match(f).group(0)):].replace('.pdf','').replace('_',' ')[:55]
            print(f"    [{arxiv_id}] {title_part}")

    if trend_reports:
        print(f"\n  [동향보고서]")
        for f in trend_reports:
            print(f"    - {f}")

    print(f"\n{'─'*65}")
    print("  ChromaDB 재구축 셀 실행 → 모든 PDF가 RAG 지식베이스에 반영됩니다.")
    print(f"{'='*65}")


show_data_inventory()


 데이터 인벤토리 현황 (data/manual_reports/)

 총 PDF     : 78건
   ├── arXiv 논문    : 51건
   └── 동향보고서    : 27건

  [arXiv 논문]
    [2103.04838] Machine-learning based methodologies for 3d x-ray 
    [2503.03182] Enhancing Memory Efficiency in Large Language Mode
    [2503.04049] Neural Network Surrogate Model for Junction Temper
    [2507.14000] Photonic Fabric Platform for AI Accelerators
    [2508.06079] Panel-Scale Reconfigurable Photonic Interconnects 
    [2508.10691] THERMOS Thermally-Aware Multi-Objective Scheduling
    [2508.19309] Leveraging 3D Technologies for Hardware Security O
    [2509.13710] CompAir Synergizing Complementary PIMs and In-Tran
    [2510.03943] 3D Electronic-Photonic Heterogenous Interconnect P
    [2510.05245] Stratum System-Hardware Co-Design with Tiered Mono
    [2510.06513] On-Package Memory with Universal Chiplet Interconn
    [2510.24113] Taming the Tail NoI Topology Synthesis for Mixed D
    [2511.12286] Sangam Chiplet-Based DRAM-PIM Accelerator with CXL
    [26

## [벡터 DB 구축] ChromaDB 재구축

`data/manual_reports/` 폴더의 모든 PDF(arXiv 논문 + 동향보고서)를 청킹하여 ChromaDB 벡터 DB를 구축합니다.  
새로운 PDF를 폴더에 넣은 뒤 아래 셀을 실행하면 지식베이스가 갱신됩니다.

| 파라미터 | 값 |
|---|---|
| 임베딩 모델 | `BAAI/bge-m3` (HuggingFace 로컬) |
| 청크 크기 | 1000 토큰 / overlap 150 |
| 재구축 여부 | `REBUILD_FROM_SCRATCH = True` 시 기존 DB 삭제 후 재생성 |


In [ ]:
# ─── arXiv 파일명 → 카테고리 키워드 매핑 ──────────────────────
KEYWORD_CATEGORY_MAP = [
    (["HBM", "DRAM", "SRAM", "Memory", "MRAM", "ReRAM", "NAND", "Flash"],         "Memory_Technology"),
    (["Chiplet", "Packaging", "Stacking", "Interconnect", "Bonding", "Heterogeneous"],"Advanced_Packaging"),
    (["PIM", "CXL", "In-Memory", "Near-Memory", "Processing-in"],                   "PIM_CXL_Computing"),
    (["Photonic", "Photonics", "Optical", "Optics"],                               "Silicon_Photonics"),
    (["Thermal", "Heat", "Cooling", "Temperature"],                                "Thermal_Management"),
    (["Neural", "NPU", "Accelerator", "GPU", "TPU", "Inference", "LLM", "Transformer"],"AI_Accelerator"),
    (["GAA", "FinFET", "CMOS", "Logic", "Node", "Foundry", "Process", "EUV"],      "Logic_Process"),
    (["GaN", "SiC", "Power", "MOSFET", "Compound"],                               "Power_Semiconductor"),
    (["Neuromorphic", "Spiking", "Memristor"],                                     "Neuromorphic"),
    (["Graphene", "2D", "Material", "Novel"],                                      "Novel_Materials"),
]


def detect_arxiv_category(filename: str) -> str:
    """파일명 키워드 → 카테고리 자동 분류"""
    name_upper = filename.upper()
    for keywords, category in KEYWORD_CATEGORY_MAP:
        if any(kw.upper() in name_upper for kw in keywords):
            return category
    return "General_Semiconductor"


def load_and_chunk_pdfs(pdf_dir: str = PDF_DIR):
    """
    manual_reports/ 의 PDF 전체를 로드·청킹합니다.
    - arXiv ID 패턴(XXXX.XXXXX_) → document_type='paper', 카테고리 자동 분류
    - 나머지 PDF             → document_type='trend_report'
    """
    print("\nPDF 로드 및 청킹 중...")
    docs = []

    if not os.path.exists(pdf_dir):
        print(f"{pdf_dir} 폴더가 없습니다. PDF를 넣어주세요.")
        return docs

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, chunk_overlap=150,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    for filename in sorted(os.listdir(pdf_dir)):
        if not filename.endswith(".pdf"):
            continue
        filepath = os.path.join(pdf_dir, filename)
        m = ARXIV_PATTERN.match(filename)

        if m:
            arxiv_id = m.group(1)
            doc_type = "paper"
            category = detect_arxiv_category(filename)
            title    = filename[len(m.group(0)):].replace('.pdf', '').replace('_', ' ')
        else:
            arxiv_id = ""
            doc_type = "trend_report"
            category = "Trend_Report"
            title    = filename.replace('.pdf', '')

        print(f"  [{doc_type:12s}] {filename[:60]}")
        try:
            loader = PyPDFLoader(filepath)
            pages  = loader.load()
            chunks = text_splitter.split_documents(pages)
            for idx, chunk in enumerate(chunks):
                # page_content가 None이거나 공백만 있으면 임베딩 오류 → 제외
                if not isinstance(chunk.page_content, str) or not chunk.page_content.strip():
                    continue
                chunk.metadata.update({
                    "source_id":     filename,
                    "document_type": doc_type,
                    "category":      category,
                    "title":         title,
                    "arxiv_id":      arxiv_id,
                    "chunk_index":   idx,
                })
                docs.append(chunk)
        except Exception as e:
            print(f"     처리 실패: {e}")

    p = sum(1 for d in docs if d.metadata["document_type"] == "paper")
    r = sum(1 for d in docs if d.metadata["document_type"] == "trend_report")
    print(f"\n  → 총 {len(docs)}개 청크")
    print(f"     ├── 논문 청크      : {p}개")
    print(f"     └── 보고서 청크    : {r}개")
    return docs


def build_vector_db(docs, persist_dir: str = CHROMA_PERSIST_DIR,
                    rebuild: bool = True):
    """ChromaDB 벡터 DB 구축"""
    if rebuild and os.path.exists(persist_dir):
        print(f"기존 ChromaDB 삭제: {persist_dir}")
        shutil.rmtree(persist_dir)

    print("임베딩 모델 로딩 (BAAI/bge-m3) ...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    # 임베딩 전 최종 방어: None / 빈 문자열 청크 제거
    docs = [d for d in docs
            if isinstance(d.page_content, str) and d.page_content.strip()]
    print(f"{len(docs)}개 청크 색인 중 ...")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    vectorstore.persist()
    print(f"\nChromaDB 구축 완료 ({persist_dir})")
    print(f"총 벡터 수: {vectorstore._collection.count()}개")
    return vectorstore


print("함수 정의 완료: load_and_chunk_pdfs / build_vector_db")


함수 정의 완료: load_and_chunk_pdfs / build_vector_db


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ChromaDB 재구축 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# REBUILD_FROM_SCRATCH = True  → 기존 DB를 완전히 삭제 후 재생성
# REBUILD_FROM_SCRATCH = False → 이미 DB가 있으면 증분 추가 (미지원 시 강제 재구축)
REBUILD_FROM_SCRATCH = True

print("[Step 1] PDF 로드 및 청킹")
print(f"  소스 폴더: {os.path.abspath(PDF_DIR)}")
all_docs = load_and_chunk_pdfs(PDF_DIR)

# 선택 수집으로 특허 문서가 있으면 합산
if 'PATENT_DOCS' in dir() and PATENT_DOCS:
    print(f"  + 특허 문서 {len(PATENT_DOCS)}건 추가")
    all_docs.extend(PATENT_DOCS)

if not all_docs:
    print("\n⚠️  처리할 문서가 없습니다.")
    print(f"   data/manual_reports/ 폴더에 PDF를 넣어주세요: {os.path.abspath(PDF_DIR)}")
else:
    print(f"\n[Step 2] ChromaDB 구축 (총 {len(all_docs)}개 청크)")
    vectorstore = build_vector_db(all_docs, CHROMA_PERSIST_DIR, rebuild=REBUILD_FROM_SCRATCH)
    print("\n✅ 완료! agent.ipynb에서 RAG 쿼리를 실행할 수 있습니다.")


[Step 1] PDF 로드 및 청킹
  소스 폴더: /Users/gani/Desktop/ai_semi/data/manual_reports

PDF 로드 및 청킹 중...
  [trend_report] 001-007. 박재성_211호 최종.pdf
  [trend_report] 001-011. 김용주_210호 최종.pdf
  [trend_report] 001-011. 민수진_215호_최종.pdf
  [trend_report] 001-012. 민수진_208호 최종.pdf
  [trend_report] 012-024. 송근혜_215호_최종.pdf
  [trend_report] 044-053. 곽희민_212호 최종.pdf
  [trend_report] 060-068. 윤두협_218호_최종.pdf
  [trend_report] 062-073. 안후영_206호.pdf
  [trend_report] 067-079. 설성호_213호 최종.pdf
  [trend_report] 075-087. 이성준_217호_최종.pdf
  [trend_report] 098-106. 전황수_208호 최종_3.pdf
  [paper       ] 2103.04838_Machine-learning_based_methodologies_for_3d_x-ray
  [paper       ] 2503.03182_Enhancing_Memory_Efficiency_in_Large_Language_Mod


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 85 0 (offset 0)
Ignoring wrong pointing object 87 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 93 0 (offset 0)
Ignoring wrong pointing object 95 0 (offset 0)
Ignoring wrong pointing object 97 0 (offset 0)
Ignoring wrong pointing object 99 0 (offset 0)
Ignoring wrong pointing object 124 0 (offset 0)
Ignoring wrong

  [paper       ] 2503.04049_Neural_Network_Surrogate_Model_for_Junction_Tempe
  [paper       ] 2507.14000_Photonic_Fabric_Platform_for_AI_Accelerators.pdf


incorrect startxref pointer(3)
parsing for Object Streams
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x5d0 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x5d0 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x899 for key /Subtype
Multiple definitions in dictionary at byte 0x8c8 for key /BBox
Multiple definitions in dictionary at byte 0x8d8 for key /Subtype


  [paper       ] 2508.06079_Panel-Scale_Reconfigurable_Photonic_Interconnects


Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x5d0 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x899 for key /Subtype
Multiple definitions in dictionary at byte 0x8c8 for key /BBox
Multiple definitions in dictionary at byte 0x8d8 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x5d0 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x899 for key /Subtype
Multiple definitions in dictionary at byte 0x8c8 for key /BBox
Multiple definitions in dictionary at byte 0x8d8 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions in dictionary at byte 0x5d0 for key /Subtype
Multiple definitions in dictionary at byte 0x7a for key /Subtype
Multiple definitions i

  [paper       ] 2508.10691_THERMOS_Thermally-Aware_Multi-Objective_Schedulin
  [paper       ] 2508.19309_Leveraging_3D_Technologies_for_Hardware_Security_
  [paper       ] 2509.13710_CompAir_Synergizing_Complementary_PIMs_and_In-Tra
  [paper       ] 2510.03943_3D_Electronic-Photonic_Heterogenous_Interconnect_
  [paper       ] 2510.05245_Stratum_System-Hardware_Co-Design_with_Tiered_Mon
  [paper       ] 2510.06513_On-Package_Memory_with_Universal_Chiplet_Intercon
  [paper       ] 2510.24113_Taming_the_Tail_NoI_Topology_Synthesis_for_Mixed_
  [paper       ] 2511.12286_Sangam_Chiplet-Based_DRAM-PIM_Accelerator_with_CX
  [paper       ] 2601.19907_RAPID-Graph_Recursive_All-Pairs_Shortest_Paths_Us
  [paper       ] 2601.19908_CHIME_Chiplet-based_Heterogeneous_Near-Memory_Acc
  [paper       ] 2602.10505_Scaling_Routers_with_In-Package_Optics_and_High-B
  [paper       ] 2602.18568_RPU_--_A_Reasoning_Processing_Unit.pdf
  [paper       ] 2603.03271_Virtual-Memory_Assisted_Buffer_Management_In_Ti

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

5478개 청크 색인 중 ...
